# ML Ensemble + Conformal Prediction

Train a walk-forward Ridge/LightGBM ensemble and add conformal intervals.

In [ ]:
from factor_forge.data.loader import generate_synthetic_prices
from factor_forge.factors.registry import FactorRegistry
from factor_forge.ml.features import build_feature_matrix
from factor_forge.ml.ensemble import FactorEnsemble
from factor_forge.ml.walk_forward import WalkForwardSplitter

prices = generate_synthetic_prices(["A", "B", "C", "D", "E"], "2015-01-01", "2024-01-01")
registry = FactorRegistry()
registry.load_builtin()

price_factors = [
    name for name in registry.list_factors()
    if registry.get(name).category.value in {"momentum", "low_volatility"}
]
scores = registry.compute(prices, names=price_factors)
panel = build_feature_matrix(prices, scores)

splitter = WalkForwardSplitter(min_train_size=500, test_size=100, step=100)
all_preds = []
for train_idx, test_idx in splitter.split(panel):
    train = panel.loc[train_idx]
    test = panel.loc[test_idx]
    X_train, y_train = train.drop(columns=["target"]), train["target"]
    X_test = test.drop(columns=["target"])
    model = FactorEnsemble(model_type="ridge").fit(X_train, y_train)
    preds = model.predict(X_test)
    all_preds.append(preds)

print(f"Generated {len(all_preds)} walk-forward folds")